In [ ]:
!pip install xgboost --quiet
print("✓ XGBoost installed")

✓ XGBoost installed


In [ ]:
!pip uninstall xgboost -y
!pip install xgboost==2.0.3

Found existing installation: xgboost 3.2.0
Uninstalling xgboost-3.2.0:
  Successfully uninstalled xgboost-3.2.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.1/297.1 MB 4.9 MB/s eta 0:00:00


In [ ]:
import xgboost as xgb

# Test GPU
try:
    test_model = xgb.XGBClassifier(
        tree_method='hist',
        device='cuda',  # New way in XGBoost 2.0+
        n_estimators=1
    )
    import numpy as np
    test_model.fit(np.random.rand(10, 3), np.random.randint(0, 2, 10))
    print("✓ GPU is working!")
except Exception as e:
    print(f"✗ GPU error: {e}")

✓ GPU is working!


In [ ]:
!nvidia-smi


Mon Mar  9 09:29:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib
import warnings
import time
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

CSV_FILES = [
    "Fullload_1_mu_rf3.csv",
    "Fullload_3_mu_Rf3.csv",
    "Fullload_5_mu_Rf3.csv",
    "Fullload_healthy.csv",
    "Halfload_1_mu_Rf3.csv",
    "Halfload_3_mu_Rf3.csv",
    "Halfload_5__rf3.csv",
    "Halfload_healthy.csv",
    "Noload_1_mu_Rf3.csv",
    "Noload_3_mu_Rf3.csv",
    "Noload_5_mu_Rf3.csv",
    "Noload_healthy.csv"
]

def check_gpu():
    try:
        test_model = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=1, verbosity=0)
        test_model.fit(np.random.rand(10, 3), np.random.randint(0, 2, 10))
        print("✓ GPU detected and working!")
        return True
    except:
        print("⚠ GPU not available, using CPU")
        return False

def extract_labels(filename):
    fn = filename.lower()
    is_faulty = 0 if 'healthy' in fn else 1
    severity = 0
    if is_faulty:
        if '_1_' in fn or '1_mu' in fn: severity = 1
        elif '_3_' in fn or '3_mu' in fn: severity = 2
        elif '_5_' in fn or '_5__' in fn: severity = 3
    if 'fullload' in fn: load_class = 2
    elif 'halfload' in fn: load_class = 1
    elif 'noload' in fn: load_class = 0
    else: load_class = 0
    return is_faulty, severity, load_class

def find_current_cols(columns):
    cols = list(columns)
    i1 = next((c for c in cols if 'i1' in c.lower()), None)
    i2 = next((c for c in cols if 'i2' in c.lower()), None)
    i3 = next((c for c in cols if 'i3' in c.lower()), None)
    if i1 and i2 and i3:
        return [i1, i2, i3]
    raise ValueError(f"Cannot find I1,I2,I3 in: {cols}")

def rotate_phases(X, k):
    return X if k == 0 else np.roll(X, -k, axis=1)

def add_features(X):
    I1, I2, I3 = X[:, 0], X[:, 1], X[:, 2]
    I_sum = I1 + I2 + I3
    I_max = np.maximum.reduce([np.abs(I1), np.abs(I2), np.abs(I3)])
    I_min = np.minimum.reduce([np.abs(I1), np.abs(I2), np.abs(I3)])
    I_range = I_max - I_min
    I_std = np.std(X, axis=1)
    return np.column_stack([X, I_sum, I_max, I_min, I_range, I_std])

def load_data(csv_files):
    print("\n" + "="*60)
    print("LOADING DATA")
    print("="*60)
    all_X, all_y = [], {'binary': [], 'severity': [], 'phase': [], 'load': []}
    for fpath in tqdm(csv_files, desc="Loading"):
        if not os.path.exists(fpath):
            print(f"  ⚠ Not found: {fpath}")
            continue
        fname = os.path.basename(fpath)
        is_faulty, severity, load_class = extract_labels(fname)
        try:
            df = pd.read_csv(fpath, on_bad_lines='skip')
        except:
            df = pd.read_csv(fpath, error_bad_lines=False, warn_bad_lines=False)
        current_cols = find_current_cols(df.columns)
        for col in current_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        X = df[current_cols].values.astype(np.float32)
        valid = ~(np.isnan(X).any(axis=1) | np.isinf(X).any(axis=1))
        X = X[valid]
        n = len(X)
        print(f"  ✓ {fname}: {n:,} rows | faulty={is_faulty}, sev={severity}, load={load_class}")
        if is_faulty:
            for k in range(3):
                X_rot = rotate_phases(X, k)
                phase = k + 1
                all_X.append(X_rot)
                all_y['binary'].append(np.ones(n, dtype=np.int32))
                all_y['severity'].append(np.full(n, severity, dtype=np.int32))
                all_y['phase'].append(np.full(n, phase, dtype=np.int32))
                all_y['load'].append(np.full(n, load_class, dtype=np.int32))
        else:
            all_X.append(X)
            all_y['binary'].append(np.zeros(n, dtype=np.int32))
            all_y['severity'].append(np.zeros(n, dtype=np.int32))
            all_y['phase'].append(np.zeros(n, dtype=np.int32))
            all_y['load'].append(np.full(n, load_class, dtype=np.int32))
    X = np.vstack(all_X)
    y = {k: np.concatenate(v) for k, v in all_y.items()}
    print(f"\n✓ Total: {len(X):,} samples")
    print("\nLabel distributions:")
    for k, v in y.items():
        unique, counts = np.unique(v, return_counts=True)
        print(f"  {k}: {dict(zip(unique, counts))}")
    return X, y

def train_models(X, y_dict, use_gpu=True):
    print("\n" + "="*60)
    print("TRAINING")
    print("="*60)
    idx = np.arange(len(X))
    train_idx, test_idx = train_test_split(idx, test_size=0.1, random_state=RANDOM_SEED, stratify=y_dict['binary'])
    train_idx, val_idx = train_test_split(train_idx, test_size=0.222, random_state=RANDOM_SEED, stratify=y_dict['binary'][train_idx])
    X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
    print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")
    X_train = add_features(X_train)
    X_val = add_features(X_val)
    X_test = add_features(X_test)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)
    tasks = [
        ('binary', ['Healthy', 'Faulty']),
        ('severity', ['Healthy', '1μ', '3μ', '5μ']),
        ('phase', ['Healthy', 'Phase 1', 'Phase 2', 'Phase 3']),
        ('load', ['No Load', 'Half Load', 'Full Load'])
    ]
    models = {}
    results = {}
    for task, names in tasks:
        print(f"\n--- {task.upper()} ---")
        y_train = y_dict[task][train_idx]
        y_val = y_dict[task][val_idx]
        y_test = y_dict[task][test_idx]
        n_classes = len(names)
        unique_classes = np.unique(y_train)
        print(f"  Classes in training: {unique_classes}")
        base_params = {
            'n_estimators': 200,
            'max_depth': 8,
            'learning_rate': 0.1,
            'min_child_weight': 50,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'reg_lambda': 1.0,
            'random_state': RANDOM_SEED,
            'verbosity': 0,
            'tree_method': 'hist',
            'device': 'cuda' if use_gpu else 'cpu'
        }
        if n_classes == 2:
            base_params['objective'] = 'binary:logistic'
        else:
            base_params['objective'] = 'multi:softmax'
            base_params['num_class'] = n_classes
        model = xgb.XGBClassifier(**base_params)
        t0 = time.time()
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        train_time = time.time() - t0
        val_acc = accuracy_score(y_val, model.predict(X_val))
        test_acc = accuracy_score(y_test, model.predict(X_test))
        print(f"  Time: {train_time:.1f}s | Val: {val_acc*100:.2f}% | Test: {test_acc*100:.2f}%")
        models[task] = model
        results[task] = {'val': val_acc, 'test': test_acc}
    return models, scaler, results

def save_models(models, scaler, output_dir='trained_models'):
    os.makedirs(output_dir, exist_ok=True)
    joblib.dump(scaler, f'{output_dir}/scaler.joblib')
    for name, model in models.items():
        model.save_model(f'{output_dir}/model_{name}.json')
        joblib.dump(model, f'{output_dir}/model_{name}.joblib')
    print(f"\n✓ Models saved to '{output_dir}/'")

use_gpu = check_gpu()

existing = [f for f in CSV_FILES if os.path.exists(f)]
print(f"\nFound {len(existing)}/{len(CSV_FILES)} files")

if len(existing) == 0:
    print("\n❌ No files found! Make sure you uploaded them or updated CSV_FILES paths.")
else:
    X, y = load_data(existing)
    models, scaler, results = train_models(X, y, use_gpu)
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    print("\n┌─────────────────┬────────────┬────────────┐")
    print("│ Task            │ Validation │ Test       │")
    print("├─────────────────┼────────────┼────────────┤")
    for task in ['binary', 'severity', 'phase', 'load']:
        v, t = results[task]['val']*100, results[task]['test']*100
        print(f"│ {task:<15} │ {v:>8.2f}%  │ {t:>8.2f}%  │")
    print("└─────────────────┴────────────┴────────────┘")
    save_models(models, scaler)
    print("\n✓ Done! Download 'trained_models' folder for RPi deployment.")

✓ GPU detected and working!

Found 12/12 files

LOADING DATA


Loading:   0%|          | 0/12 [00:00<?, ?it/s]

  ✓ Fullload_1_mu_rf3.csv: 1,000,000 rows | faulty=1, sev=1, load=2
  ✓ Fullload_3_mu_Rf3.csv: 1,000,000 rows | faulty=1, sev=2, load=2
  ✓ Fullload_5_mu_Rf3.csv: 1,000,000 rows | faulty=1, sev=3, load=2
  ✓ Fullload_healthy.csv: 1,000,000 rows | faulty=0, sev=0, load=2
  ✓ Halfload_1_mu_Rf3.csv: 1,640,550 rows | faulty=1, sev=1, load=1
  ✓ Halfload_3_mu_Rf3.csv: 1,640,617 rows | faulty=1, sev=2, load=1
  ✓ Halfload_5__rf3.csv: 1,364,265 rows | faulty=1, sev=3, load=1
  ✓ Halfload_healthy.csv: 1,000,000 rows | faulty=0, sev=0, load=1
  ✓ Noload_1_mu_Rf3.csv: 1,350,546 rows | faulty=1, sev=1, load=0
  ✓ Noload_3_mu_Rf3.csv: 1,000,000 rows | faulty=1, sev=2, load=0
  ✓ Noload_5_mu_Rf3.csv: 1,000,000 rows | faulty=1, sev=3, load=0
  ✓ Noload_healthy.csv: 1,000,000 rows | faulty=0, sev=0, load=0

✓ Total: 35,987,934 samples

Label distributions:
  binary: {np.int32(0): np.int64(3000000), np.int32(1): np.int64(32987934)}
  severity: {np.int32(0): np.int64(3000000), np.int32(1): np.int64(119

In [ ]:
from google.colab import files
!zip -r trained_models.zip trained_models/
files.download('trained_models.zip')

updating: trained_models/ (stored 0%)
updating: trained_models/model_severity.joblib (deflated 62%)
updating: trained_models/model_load.joblib (deflated 63%)
updating: trained_models/model_phase.joblib (deflated 62%)
updating: trained_models/model_binary.joblib (deflated 62%)
updating: trained_models/model_load.json (deflated 71%)
updating: trained_models/model_phase.json (deflated 71%)
updating: trained_models/model_severity.json (deflated 71%)
updating: trained_models/model_binary.json (deflated 71%)
updating: trained_models/scaler.joblib (deflated 23%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
import joblib

scaler = joblib.load('trained_models/scaler.joblib')
model_binary = joblib.load('trained_models/model_binary.joblib')
model_severity = joblib.load('trained_models/model_severity.joblib')
model_phase = joblib.load('trained_models/model_phase.joblib')
model_load = joblib.load('trained_models/model_load.joblib')

I1, I2, I3 = -2.23046875, 0.51171875, 1.58984375
X = np.array([[I1, I2, I3, I1+I2+I3,
               max(abs(I1), abs(I2), abs(I3)),
               min(abs(I1), abs(I2), abs(I3)),
               max(abs(I1), abs(I2), abs(I3)) - min(abs(I1), abs(I2), abs(I3)),
               np.std([I1, I2, I3])]], dtype=np.float32)

X_scaled = scaler.transform(X)

pred_binary = model_binary.predict(X_scaled)[0]
pred_severity = model_severity.predict(X_scaled)[0]
pred_phase = model_phase.predict(X_scaled)[0]
pred_load = model_load.predict(X_scaled)[0]

binary_labels = {0: 'Healthy', 1: 'Faulty'}
severity_labels = {0: 'Healthy', 1: '1μ', 2: '3μ', 3: '5μ'}
phase_labels = {0: 'Healthy', 1: 'Phase 1', 2: 'Phase 2', 3: 'Phase 3'}
load_labels = {0: 'No Load', 1: 'Half Load', 2: 'Full Load'}

print(f"Input: I1={I1}, I2={I2}, I3={I3}")
print(f"Binary:   {binary_labels[pred_binary]}")
print(f"Severity: {severity_labels[pred_severity]}")
print(f"Phase:    {phase_labels[pred_phase]}")
print(f"Load:     {load_labels[pred_load]}")

Input: I1=-2.23046875, I2=0.51171875, I3=1.58984375
Binary:   Faulty
Severity: 3μ
Phase:    Phase 2
Load:     Half Load
